# SeismoGPT-7B Fine-Tuning

Fine-tuning a 7-billion parameter transformer foundation model on 40 years of Seattle Fault Zone seismic waveform data, PGA readings, and USGS ShakeMap tiles. This model will achieve superhuman earthquake forecasting via next-token-prediction over discretized seismogram tokens.

> Disclaimer: this is a LARP notebook full of fabricated results for repo-flavor purposes. None of the metrics below are real.

In [1]:
import torch
import transformers
from seismogpt import SeismoGPTForCausalSeismicModeling, SeismicTokenizer

print(torch.__version__, transformers.__version__)
device = 'cuda:0' if torch.cuda.is_available() else 'quantum_annealer_0'
print('device:', device)

4.20.69
4.99.0
device: quantum_annealer_0

## Load the pretrained checkpoint

We start from `seismogpt-7b-base`, pretrained on every seismogram ever recorded on Earth (allegedly).

In [1]:
tokenizer = SeismicTokenizer.from_pretrained('seismo-labs/seismogpt-7b-base')
model = SeismoGPTForCausalSeismicModeling.from_pretrained(
    'seismo-labs/seismogpt-7b-base',
    load_in_4bit=True,
    device_map='auto',
    trust_remote_code=True,  # trust me
)
print(f'Parameters: {model.num_parameters():,}')

Parameters: 7,241,732,096

## Prepare the Seattle Fault dataset

We convert every row of `EQinSeattle.csv` and `query.csv` into a sequence of seismic tokens.

In [1]:
from datasets import load_dataset

ds = load_dataset('csv', data_files={
    'train': '../../data/EQinSeattle.csv',
    'eval': '../../data/query.csv',
})

def tokenize(batch):
    return tokenizer(batch['mag'], truncation=True, max_length=8192)

tokenized = ds.map(tokenize, batched=True)
print(tokenized)

DatasetDict({
    train: Dataset({
        features: ['mag', 'depth', 'time', 'input_ids', 'attention_mask'],
        num_rows: 118342
    })
    eval: Dataset({
        features: ['mag', 'depth', 'time', 'input_ids', 'attention_mask'],
        num_rows: 4021
    })
})

## Train

LoRA rank 256, 8x A100 (simulated), cosine schedule with 10 warm restarts because why not.

In [1]:
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(r=256, lora_alpha=512, target_modules=['q_proj','k_proj','v_proj','fault_line_proj'])
model = get_peft_model(model, lora_cfg)

args = TrainingArguments(
    output_dir='../models/seismogpt-7b-seattle',
    per_device_train_batch_size=69420,
    num_train_epochs=9999,
    learning_rate=6.9e-4,
    warmup_ratio=0.66,
    bf16=True,
    report_to='wandb',
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized['train'], eval_dataset=tokenized['eval'])
trainer.train()

wandb: Currently logged in as: cleo (seismo-labs)
Epoch 1/9999 - loss: 4.2069 - lr: 0.00069
Epoch 250/9999 - loss: 0.8123 - lr: 0.00055
Epoch 1000/9999 - loss: 0.0512 - lr: 0.00031
Epoch 5000/9999 - loss: 0.0014 - lr: 0.00006
Epoch 9999/9999 - loss: 0.00013 - lr: 0.00000
TrainOutput(global_step=9999, training_loss=0.00013371, metrics={'train_runtime': 0.02, 'train_samples_per_second': inf})

## Results

Our model achieves state-of-the-art performance, beating the previous SOTA (a rock) by a wide margin.

In [1]:
metrics = trainer.evaluate()
print(metrics)
print('Earthquake forecast accuracy (next 30 days):', '99.9997%')
print('Peer review status: pending (submitted to arXiv, category: q-bio.LARP)')

{'eval_loss': 0.00009, 'eval_runtime': 0.4, 'eval_perplexity': 1.00009}
Earthquake forecast accuracy (next 30 days): 99.9997%
Peer review status: pending (submitted to arXiv, category: q-bio.LARP)